# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library, referencing all entities by their `@id` attributes per best practice.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples, collected via mass spectrometry.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

We'll print out the details for each record set, its fields, and their `@id`s.


In [ ]:
# Display an overview of all record sets and their fields by @id
record_set_ids = []
print("Record sets available in dataset:")
for rs in metadata.record_sets:
    print(f"- Record set: {rs['@id']} (name: {rs['name'] if 'name' in rs else 'N/A'})")
    record_set_ids.append(rs['@id'])
    # Fields/columns of this record set
    if 'fields' in rs:
        print("  Fields:")
        for f in rs['fields']:
            field_id = f['@id'] if isinstance(f, dict) and '@id' in f else str(f)
            field_name = f.get('name', 'N/A') if isinstance(f, dict) else 'N/A'
            print(f"    - {field_id} (name: {field_name})")


## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis using their `@id`s. Review the previous cell's output to select which sets to extract.


In [ ]:
# Example: Extract data from available record sets
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if not records:
            print(f"[WARN] Record set {record_set_id} returned no records.")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id}.")
        print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Error loading records from {record_set_id}: {str(e)}")
if not dataframes:
    print("No data frames could be loaded. Check available data sources in metadata.")

## 4. Exploratory Data Analysis (EDA)
Let's perform some typical EDA: filtering, normalizing, and grouping. All columns and fields should be referenced with their `@id` as available.

_Update `selected_record_set_id`, `numeric_field_id`, and `group_field_id` as appropriate for your dataset!_


In [ ]:
# ---
# Example selection: Replace these IDs with values from the OVERVIEW!
# ---
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]
    print(f"Using record set: {selected_record_set_id}")
    print(f"Available columns: {df.columns.tolist()}")

    # Attempt to select a numeric field automatically
    numeric_field_candidates = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean()  # e.g., filter on mean
        filtered_df = df[df[numeric_field_id] > threshold].copy()

        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std())

        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by the first object-type field (could be e.g. protein type, accession, etc.)
        group_field_candidates = df.select_dtypes(include=['object']).columns.tolist()
        if group_field_candidates:
            group_field_id = group_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    else:
        print("No numeric fields identified in this record set for EDA.")
else:
    print("No extracted data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields. Update the field IDs as necessary for your analysis.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution (if available)
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]
    numeric_field_candidates = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.show()

        # If there's a group field (object column), visualize boxplot
        group_field_candidates = df.select_dtypes(include=['object']).columns.tolist()
        if group_field_candidates:
            group_field_id = group_field_candidates[0]
            if df[group_field_id].value_counts().shape[0] < 20:  # avoid too many groups
                plt.figure(figsize=(10,5))
                sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
                plt.title(f"{numeric_field_id} by {group_field_id}")
                plt.show()
    else:
        print("No numeric fields available for visualization.")
else:
    print("No data available for visualization.")

## 6. Conclusion

In this notebook, we loaded the FAIR^2 dataset using the `mlcroissant` library, referenced all fields and record sets by their `@id`, and demonstrated exploratory data analysis and basic visualizations. This approach provides a reproducible workflow for working with complex machine-actionable metadata and supporting large, structured research datasets.

You can now extend this notebook to perform richer analyses using additional fields and domain-specific questions!